In [39]:
import pandas as pd
import numpy as np
import nltk
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [40]:


nltk.download('stopwords')
stop_words = set(nltk.corpus.stopwords.words('english'))



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\drago\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [41]:
def load_data(file_path):
    data = pd.read_csv(file_path, encoding='latin1')
    return data


In [43]:

def clean_data(data):
    data['text'] = data['text'].fillna('')
    data['sentiment'] = data['sentiment'].fillna('neutral')
    data['sentiment'] = data['sentiment'].astype(str)
    data['text'] = data['text'].apply(lambda x: " ".join([word for word in x.split() if word not in stop_words]))
    return data

In [44]:


class SentimentAnalysisDataset(Dataset):
    def __init__(self, texts, labels, vectorizer):
        self.features = vectorizer.transform(texts).toarray()
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        features = torch.tensor(self.features[index], dtype=torch.float32)
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return features, label


In [45]:

class SentimentClassifier(nn.Module):
    def __init__(self, input_size, output_size):
        super(SentimentClassifier, self).__init__()
        self.layer1 = nn.Linear(input_size, 512)
        self.dropout1 = nn.Dropout(0.5)
        self.layer2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(0.5)
        self.output_layer = nn.Linear(256, output_size)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.dropout1(x)
        x = torch.relu(self.layer2(x))
        x = self.dropout2(x)
        x = self.output_layer(x)
        return x


In [46]:

def train_model(model, loss_function, optimizer, train_loader, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        for features, labels in train_loader:
            optimizer.zero_grad()
            predictions = model(features)
            loss = loss_function(predictions, labels)
            loss.backward()
            optimizer.step()
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')


In [47]:

def evaluate_model(model, test_loader):
    model.eval()
    all_predictions, all_labels = [], []
    with torch.no_grad():
        for features, labels in test_loader:
            predictions = model(features)
            _, predicted_labels = torch.max(predictions, dim=1)
            all_predictions.extend(predicted_labels.numpy())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    report = classification_report(all_labels, all_predictions)
    return accuracy, report


In [54]:

def main(train_file, test_file):
    train_data = load_data(train_file)
    test_data = load_data(test_file)
    train_data = clean_data(train_data)
    test_data = clean_data(test_data)

    vectorizer = TfidfVectorizer(max_features=5000)
    X_train = train_data['text']
    y_train = train_data['sentiment']
    X_test = test_data['text']
    y_test = test_data['sentiment']

    vectorizer.fit(X_train)
    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)
    y_test = label_encoder.transform(y_test)

    train_dataset = SentimentAnalysisDataset(X_train, y_train, vectorizer)
    test_dataset = SentimentAnalysisDataset(X_test, y_test, vectorizer)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)

    model = SentimentClassifier(input_size=5000, output_size=len(label_encoder.classes_))
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_model(model, loss_function, optimizer, train_loader)

    accuracy, report = evaluate_model(model, test_loader)
    print(f"Accuracy: {accuracy}")
    print(f"Classification Report:\n{report}")



In [ ]:
main("trainsenti.csv", "testsenti.csv")


Epoch 1/10, Loss: 0.8068807721138
Epoch 2/10, Loss: 0.7577828764915466
Epoch 3/10, Loss: 0.48552292585372925
Epoch 4/10, Loss: 0.40742233395576477
Epoch 5/10, Loss: 0.10280992835760117
Epoch 6/10, Loss: 0.1825784146785736
Epoch 7/10, Loss: 0.011965501122176647
Epoch 8/10, Loss: 0.0160834938287735
